# 📝 Day 2 Assignments — FastAPI Routing

---

Build a small **books** API. Each task adds an endpoint. We'll use `TestClient` to verify everything from inside the notebook.


In [ ]:
!pip install fastapi uvicorn


In [ ]:
# Shared setup — re-run after editing endpoints
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI(title="Day 2 Assignments - Books API")

# In-memory "database"
BOOKS: list[dict] = [
    {"id": 1, "title": "Clean Code",     "author": "bob"},
    {"id": 2, "title": "The Pragmatic Programmer", "author": "alice"},
    {"id": 3, "title": "Refactoring",   "author": "alice"},
]

client = TestClient(app)


## Task 1 — `GET /books`

**Problem:** Return the full list of books.

**Expected output:**
```
[{"id":1,...}, {"id":2,...}, {"id":3,...}]
```

💡 **Hint:** just `return BOOKS`.


In [ ]:
@app.get("/books")
def list_books(author: str | None = None):
    # Task 4 lives here too — filter by author if provided
    if author is not None:
        return [b for b in BOOKS if b["author"] == author]
    return BOOKS

print(client.get("/books").json())


## Task 2 — `GET /books/{book_id}`

**Problem:** Return a single book by id. If not found, raise `HTTPException(404)`.

**Expected output:**
```
200 -> {"id":1,...}
404 -> {"detail":"book not found"}
```

💡 **Hint:** `next((b for b in BOOKS if b['id']==book_id), None)`.


In [ ]:
@app.get("/books/{book_id}")
def get_book(book_id: int):
    book = next((b for b in BOOKS if b["id"] == book_id), None)
    if book is None:
        raise HTTPException(status_code=404, detail="book not found")
    return book

print(client.get("/books/1").status_code, client.get("/books/1").json())
print(client.get("/books/999").status_code, client.get("/books/999").json())


## Task 3 — `POST /books`

**Problem:** Accept a JSON body and append a new book. Auto-assign the `id`.

**Expected output:**
```
201/200 -> {"id": 4, "title": "...", "author": "..."}
```

💡 **Hint:** `max(b['id'] for b in BOOKS) + 1`.


In [ ]:
@app.post("/books")
def create_book(book: dict):
    new_id = (max((b["id"] for b in BOOKS), default=0)) + 1
    book["id"] = new_id
    BOOKS.append(book)
    return book

print(client.post("/books", json={"title": "Domain-Driven Design", "author": "eric"}).json())
print(client.get("/books").json())


## Task 4 — Filter by author (query param)

**Problem:** Extend `GET /books` to accept an optional `?author=alice` query param.

**Expected output:**
```
client.get("/books?author=alice")  -> only Alice's books
```

💡 **Hint:** already implemented in Task 1's handler — just call it.


In [ ]:
print(client.get("/books?author=alice").json())
print(client.get("/books?author=ghost").json())


## 🎁 Bonus — PUT + DELETE, full CRUD check

**Problem:** Add `PUT /books/{id}` (replace) and `DELETE /books/{id}`. Then write **four** `TestClient` calls that verify the full CRUD cycle: create → read → update → delete.

💡 **Hint:** for DELETE return a 204 with empty body, or 200 with `{"ok": True}`.


In [ ]:
@app.put("/books/{book_id}")
def replace_book(book_id: int, book: dict):
    for i, b in enumerate(BOOKS):
        if b["id"] == book_id:
            book["id"] = book_id
            BOOKS[i] = book
            return book
    raise HTTPException(status_code=404, detail="book not found")

@app.delete("/books/{book_id}")
def delete_book(book_id: int):
    for i, b in enumerate(BOOKS):
        if b["id"] == book_id:
            BOOKS.pop(i)
            return {"ok": True}
    raise HTTPException(status_code=404, detail="book not found")


In [ ]:
# Full CRUD round-trip
created = client.post("/books", json={"title": "Test", "author": "me"}).json()
bid = created["id"]
print("CREATE:", created)

print("READ:  ", client.get(f"/books/{bid}").json())

updated = client.put(f"/books/{bid}", json={"title": "Test v2", "author": "me"}).json()
print("UPDATE:", updated)

print("DELETE:", client.delete(f"/books/{bid}").json())
print("GONE?: ", client.get(f"/books/{bid}").status_code)


---

✅ **Done!** You've built a full CRUD REST API in ~50 lines of Python.
